In [8]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from glob import glob
import torchaudio
import torchinfo
import torch
import torch.nn.functional as F
#from core.nn.basic_dataloader import *
from tqdm import tqdm
import math

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
mouseFLAC  = glob("vctk/gen/flac/wav48_silence_trimmed/*/*.flac")
stockFLAC = [m_fl.replace("vctk/gen/flac", "vctk/stock") for m_fl in mouseFLAC]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: {}".format(device))
torch.set_default_device(device)

device: cuda


In [6]:
Fs=16000
maxLen = 2*131072
def loadFLAC(flac_fn):
    flacData, sr = torchaudio.load(flac_fn)
    # resample to Fs
    if (sr != Fs) :
        flacData = torchaudio.transforms.Resample(sr, Fs)(flacData)
    flacData = flacData[0][1:]
    flacData = F.pad(flacData, (0, maxLen - flacData.shape[-1]))

    return flacData.to(device)

#flacData, sr = torchaudio.load(mouseFLAC[0])
#plt.plot(flacData[0])


In [9]:
batch_size=4

# nominally 3832 // 4, but GPU memory limit
max_batch_count=64
batch_count=min(max_batch_count, (len(stockFLAC)//4))

f_set = torch.zeros([batch_count, 4,1, maxLen])
batch=0
for i in tqdm(range(0, batch_count-1)):
    for j in range(batch_size):
        #print(4*(len(stockFLAC)//4))

        #print(i*4+j)
        
        f = loadFLAC(stockFLAC[i*4+j])
        #f.reshape(1,1,len(f))

        f_set[batch][j][0] = f

    #print(i)


    batch+=1

###
###
### MUST CONVERT ALL INPUTS TO POWER-OF-TWO LENGTH
###
###

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:01<00:00, 35.62it/s]


In [10]:
    
noise_idx = 1

# f=loadFLAC(stockFLAC[0])
f_noisy=loadFLAC(mouseFLAC[noise_idx])
f_noisy=f_noisy.reshape(1,1,len(f_noisy))
f_noisy_path=mouseFLAC[noise_idx]
f_clean_path=f_noisy_path.replace("vctk/gen/flac", "vctk/stock")

print(f_clean_path)

#f=f.reshape(1,1,len(f))
# print(f.shape)
# print(f.get_device())

# q=2**math.ceil(math.log2(len(f)))
# print(q)

# #f=F.pad(f, (0, maxLen - f.shape[-1]))

# print(f.shape)
# print(f_noisy.shape)


vctk/stock/wav48_silence_trimmed/p240/p240_240_mic2.flac


In [11]:
f_set[0].shape

torch.Size([4, 1, 262144])

In [12]:
from audio_diffusion_pytorch import DiffusionModel, UNetV0, VDiffusion, VSampler
import torch.optim as optim

model = DiffusionModel(
    net_t=UNetV0, # The model type used for diffusion (U-Net V0 in this case)
    in_channels=1, # U-Net: number of input/output (audio) channels
    channels=[8, 32, 64, 128, 256, 512, 512, 1024, 1024], # U-Net: channels at each layer
    factors=[1, 4, 4, 4, 2, 2, 2, 2, 2], # U-Net: downsampling and upsampling factors at each layer
    items=[1, 2, 2, 2, 2, 2, 2, 4, 4], # U-Net: number of repeating items at each layer
    attentions=[0, 0, 0, 0, 0, 1, 1, 1, 1], # U-Net: attention enabled/disabled at each layer
    attention_heads=8, # U-Net: number of attention heads per attention item
    attention_features=64, # U-Net: number of attention features per attention item
    diffusion_t=VDiffusion, # The diffusion method used
    sampler_t=VSampler, # The diffusion sampler used
)


# Train model with audio waveforms



epoch_count = 10

epochs = tqdm(range(epoch_count), desc="set")
training_progress = tqdm(total = max_batch_count, desc="progress:", position=1, leave=True)


for e in epochs:
    training_progress.set_description("epoch " + str(e))
    training_progress.reset()
    for i in range(max_batch_count):
        audio = f_set[i] # [batch_size, in_channels, length]
        loss = model(audio)
        loss.backward()
        training_progress.update()


set: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [09:42<00:00, 58.25s/it]


In [13]:
torch.save(model.state_dict(), "diffusion.pt")

In [14]:

print(f_noisy.shape)

# Turn noise into new audio sample with diffusion
noise = f_noisy # [batch_size, in_channels, length]

rep_ct=40

#for i in tqdm(range(rep_ct)):
noise = model.sample(noise, num_steps=210) # Suggested num_steps 10-100

sample = noise

print("sample created")

torch.Size([1, 1, 262144])
sample created


In [15]:
import noisereduce as nr

save_name="test.flac"
print(sample[0][0].shape)

#q = nr.reduce_noise(sample[0][0].to("cpu").unsqueeze(0).clone().detach(), Fs)
#print(torch.tensor(q))
#torchaudio.save(save_name,torch.tensor(q).to("cpu"), Fs)

torchaudio.save(save_name,sample[0][0].to("cpu").unsqueeze(0).clone().detach(), Fs)

torch.Size([262144])


In [16]:
print("saved output flac at {}".format(save_name))
print("  | compare with noisy input: {}".format(f_noisy_path))
print("  | compare with clean input: {}".format(f_clean_path))

saved output flac at test.flac
  | compare with noisy input: vctk/gen/flac/wav48_silence_trimmed/p240/p240_240_mic2.flac
  | compare with clean input: vctk/stock/wav48_silence_trimmed/p240/p240_240_mic2.flac


In [ ]:
#plt.plot(torch.tensor(q).to("cpu"))

data=q

times = np.linspace(0,len(data),len(data))/Fs
plt.plot(times,data)
#plt.xlim(show.t_min, show.t_max)
#plt.ylim(-.1,.1)
plt.ylabel('Time (s)')
plt.show()